# Lesson 5A: K-Nearest Neighbors Theory

## Table of Contents1. Introduction2. Distance Metrics3. Curse of Dimensionality4. KD-Tree5. Bias-Variance6. Implementations7. Applications

## IntroductionK-Nearest Neighbors (KNN) is a simple yet powerful lazy learning algorithm. The core idea: to predict a new point's label, examine its k nearest neighbors in training data and use majority vote.Key characteristics:- Instance-based learning- No parametric assumptions- Non-parametric- Locally adaptive- Interpretable

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import load_iris, make_classificationfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_splitimport timeimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8-darkgrid')sns.set_palette('husl')np.random.seed(42)print('Libraries imported!')

## Distance MetricsThe foundation of KNN lies in distance calculation.

In [ ]:
# Distance metric implementationsdef euclidean_distance(x1, x2):    return np.sqrt(np.sum((x1 - x2) ** 2))def manhattan_distance(x1, x2):    return np.sum(np.abs(x1 - x2))def chebyshev_distance(x1, x2):    return np.max(np.abs(x1 - x2))def minkowski_distance(x1, x2, p=2):    return np.sum(np.abs(x1 - x2) ** p) ** (1.0 / p)# Example calculationsx1, x2 = np.array([0, 0]), np.array([3, 4])print('Distance Examples:')print(f'Euclidean: {euclidean_distance(x1, x2):.4f}')print(f'Manhattan: {manhattan_distance(x1, x2):.4f}')print(f'Chebyshev: {chebyshev_distance(x1, x2):.4f}')print(f'Minkowski(p=3): {minkowski_distance(x1, x2, p=3):.4f}')

In [ ]:
# Vectorized distance computation (essential for efficiency)def euclidean_vectorized(X, y):    X_sqnorm = np.sum(X ** 2, axis=1, keepdims=True)    y_sqnorm = np.sum(y ** 2)    X_dot_y = np.dot(X, y)    distances_sq = X_sqnorm.ravel() + y_sqnorm - 2 * X_dot_y    return np.sqrt(np.maximum(distances_sq, 0))# BenchmarkX = np.random.randn(10000, 100)y = np.random.randn(100)start = time.time()for _ in range(10):    _ = euclidean_vectorized(X, y)vec_time = time.time() - startprint(f'Vectorized 10 queries on 10000 points: {vec_time:.4f}s')

In [ ]:
# Visualize different distance metricsfig, axes = plt.subplots(1, 3, figsize=(15, 4))theta = np.linspace(0, 2*np.pi, 100)# Euclideanaxes[0].plot(5*np.cos(theta), 5*np.sin(theta), 'b-', linewidth=2)axes[0].scatter([0], [0], c='red', s=100, marker='x', zorder=5)axes[0].set_title('Euclidean (Circle)')axes[0].set_aspect('equal')axes[0].grid(True, alpha=0.3)# Manhattanmanhattan_pts = [[5, 0], [0, 5], [-5, 0], [0, -5], [5, 0]]manhattan_pts = np.array(manhattan_pts)axes[1].plot(manhattan_pts[:, 0], manhattan_pts[:, 1], 'r-', linewidth=2)axes[1].scatter([0], [0], c='red', s=100, marker='x', zorder=5)axes[1].set_title('Manhattan (Diamond)')axes[1].set_aspect('equal')axes[1].grid(True, alpha=0.3)# Chebyshevchebyshev_pts = [[5, 5], [-5, 5], [-5, -5], [5, -5], [5, 5]]chebyshev_pts = np.array(chebyshev_pts)axes[2].plot(chebyshev_pts[:, 0], chebyshev_pts[:, 1], 'g-', linewidth=2)axes[2].scatter([0], [0], c='red', s=100, marker='x', zorder=5)axes[2].set_title('Chebyshev (Square)')axes[2].set_aspect('equal')axes[2].grid(True, alpha=0.3)for ax in axes:    ax.set_xlim(-6, 6)    ax.set_ylim(-6, 6)plt.suptitle('Distance Metrics: Points at distance 5 from origin', fontweight='bold')plt.tight_layout()plt.show()

## Curse of DimensionalityAs dimensions increase, all distances become approximately equal.

In [ ]:
# Distance concentration phenomenondef distance_concentration(n_samples=1000, n_trials=3):    dimensions = np.arange(1, 51, 10)    ratios = []    for d in dimensions:        d_ratios = []        for _ in range(n_trials):            X = np.random.uniform(0, 1, (n_samples, d))            query = np.random.uniform(0, 1, d)            dist = euclidean_vectorized(X, query)            ratio = np.max(dist) / (np.min(dist) + 1e-10)            d_ratios.append(ratio)        ratios.append(np.mean(d_ratios))    return dimensions, ratiosprint('Computing distance concentration...')dims, ratios = distance_concentration()print('\nDimension-Distance Ratio:')for d, r in zip(dims, ratios):    print(f'  d={d:2d}: ratio={r:8.2f}')# Visualizefig, ax = plt.subplots(figsize=(10, 6))ax.semilogy(dims, ratios, 'o-', linewidth=2, markersize=8, color='darkred')ax.axhline(y=1, color='green', linestyle='--', linewidth=2, label='All distances equal')ax.set_xlabel('Dimensions')ax.set_ylabel('Max Distance / Min Distance (log)')ax.set_title('Distance Concentration: All Distances Become Equal in High Dimensions')ax.grid(True, alpha=0.3, which='both')ax.legend()plt.tight_layout()plt.show()

## KD-Tree: Efficient Nearest Neighbor SearchKD-trees enable O(log n) search instead of O(n) brute force.

In [ ]:
# KD-Tree Nodeclass KDTreeNode:    def __init__(self, point=None, axis=None, left=None, right=None):        self.point = point        self.axis = axis        self.left = left        self.right = right# KD-Tree classclass KDTree:    def __init__(self, X):        self.X = X        self.root = self._build(np.arange(len(X)), axis=0)        self.n_features = X.shape[1]    def _build(self, indices, axis=0):        if len(indices) == 0:            return None        axis_vals = self.X[indices, axis % self.n_features]        median_pos = np.argsort(axis_vals)[len(indices) // 2]        median_idx = indices[median_pos]        axis_val = self.X[median_idx, axis % self.n_features]        left_idx = indices[self.X[indices, axis % self.n_features] < axis_val]        right_idx = indices[self.X[indices, axis % self.n_features] >= axis_val]        return KDTreeNode(            point=self.X[median_idx],            axis=axis % self.n_features,            left=self._build(left_idx, axis + 1),            right=self._build(right_idx, axis + 1)        )    def query(self, point, k=1):        best = []        self._search(self.root, point, k, best)        return sorted(best, key=lambda x: x[1])[:k]    def _search(self, node, point, k, best):        if node is None:            return        dist = euclidean_distance(point, node.point)        if len(best) < k:            best.append((node.point.copy(), dist))            best.sort(key=lambda x: x[1])        elif dist < best[-1][1]:            best[-1] = (node.point.copy(), dist)            best.sort(key=lambda x: x[1])        axis = node.axis        if point[axis] < node.point[axis]:            near, far = node.left, node.right        else:            near, far = node.right, node.left        self._search(near, point, k, best)        if len(best) < k or abs(point[axis] - node.point[axis]) < best[-1][1]:            self._search(far, point, k, best)# Testnp.random.seed(42)X_kd = np.random.randn(100, 2) * 10tree = KDTree(X_kd)query = np.array([0.0, 0.0])neighbors = tree.query(query, k=5)print(f'Query: {query}')print(f'5 nearest neighbors:')for i, (p, d) in enumerate(neighbors, 1):    print(f'  {i}. {p}: distance={d:.4f}')

## KNN Classifier Implementation

In [ ]:
class KNNClassifier:    def __init__(self, k=5, metric='euclidean'):        self.k = k        self.metric = metric        self.X_train = None        self.y_train = None    def fit(self, X, y):        self.X_train = X.astype(np.float64)        self.y_train = y        return self    def _distances(self, X_test):        if self.metric == 'euclidean':            return euclidean_vectorized(self.X_train, X_test)        else:            return np.sum(np.abs(self.X_train - X_test), axis=1)    def predict(self, X_test):        predictions = []        for x in X_test:            dist = self._distances(x)            k_idx = np.argsort(dist)[:self.k]            labels = self.y_train[k_idx]            unique, counts = np.unique(labels, return_counts=True)            predictions.append(unique[np.argmax(counts)])        return np.array(predictions)    def score(self, X_test, y_test):        return np.mean(self.predict(X_test) == y_test)# Test on Irisiris = load_iris()X, y = iris.data, iris.targetX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)scaler = StandardScaler()X_train = scaler.fit_transform(X_train)X_test = scaler.transform(X_test)knn = KNNClassifier(k=5)knn.fit(X_train, y_train)print(f'KNN Classifier Test (Iris):')print(f'Train accuracy: {knn.score(X_train, y_train):.4f}')print(f'Test accuracy: {knn.score(X_test, y_test):.4f}')

## Bias-Variance TradeoffSmall k: low bias, high variance (overfitting)Large k: high bias, low variance (underfitting)

In [ ]:
# Test different k valuesk_values = [1, 3, 5, 7, 10, 15]results = []for k in k_values:    knn_k = KNNClassifier(k=k)    knn_k.fit(X_train, y_train)    train_acc = knn_k.score(X_train, y_train)    test_acc = knn_k.score(X_test, y_test)    results.append((k, train_acc, test_acc))print('k  | Train Acc | Test Acc')print('-' * 28)for k, train, test in results:    print(f'{k:2d} | {train:9.4f} | {test:8.4f}')# Visualizeks = [r[0] for r in results]trains = [r[1] for r in results]tests = [r[2] for r in results]fig, ax = plt.subplots(figsize=(10, 6))ax.plot(ks, trains, 'o-', linewidth=2, markersize=8, label='Training')ax.plot(ks, tests, 's-', linewidth=2, markersize=8, label='Test')ax.set_xlabel('k (number of neighbors)')ax.set_ylabel('Accuracy')ax.set_title('KNN Performance vs k')ax.legend()ax.grid(True, alpha=0.3)ax.set_xticks(ks)plt.tight_layout()plt.show()

## Complexity AnalysisBrute force: O(1) training, O(n) queryKD-tree: O(n log n) training, O(log n) query

In [ ]:
# Summaryprint('=' * 70)print('KNN SUMMARY')print('=' * 70)print('''ADVANTAGES:- Simple and intuitive- Non-parametric- No training phase- InterpretableDISADVANTAGES:- Slow prediction (O(n) brute force)- Sensitive to distance metric- Curse of dimensionality- Stores all training dataWHEN TO USE:- Small to medium datasets- Low-moderate dimensions- Non-linear boundaries needed- Interpretability importantWHEN TO AVOID:- Very large datasets- High dimensions- Real-time required- Need probabilistic outputs''')

## ConclusionKNN is a powerful, interpretable algorithm. Key considerations:- Choose k via cross-validation- Normalize features- Use KD-trees for speedup- Beware curse of dimensionality